# Subject 01 Flatmap Cross-Attention MindEye SD Bridge


# 1. Cache + Train Flatmap-To-MindEye-SD Bridge
Encode NSD images with MindEye's frozen `sd_image_var_autoenc.pth`, cache bottleneck latents once, then train the BRAID v2 flatmap cross-attention bridge plus MindEye-style latent upsampler to predict Stable Diffusion VAE latents.


## Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, glob, json, random, shutil, time, gc, sys, subprocess
from collections import defaultdict, OrderedDict

import importlib.util
_missing = []
for module_name, package_name in [("diffusers", "diffusers"), ("accelerate", "accelerate"), ("transformers", "transformers"), ("safetensors", "safetensors"), ("skimage", "scikit-image")]:
    if importlib.util.find_spec(module_name) is None:
        _missing.append(package_name)
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from diffusers import AutoencoderKL
try:
    from diffusers.models.vae import Decoder
except Exception:
    from diffusers.models.autoencoders.vae import Decoder

BASE = "/content"
DRIVE_PROJECT_DIR = f"{BASE}/drive/Shareddrives/FMRI_Paper"
INPUT_DIR = f"{DRIVE_PROJECT_DIR}/inputs"
FLATMAP_TOKENS_DIR = f"{INPUT_DIR}/flatmap_brain_tokens"
# Put an old timestamp here to resume a previous run; leave blank for a new run.
RUN_TIMESTAMP = ""  # e.g. "20260831_061019"
if not RUN_TIMESTAMP:
    RUN_TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/outputs/subject01_flatmap_mindeye_sd_bridge_{RUN_TIMESTAMP}"
SUBJECT = "subj01"
TRIALS_PER_SESSION = 750
KNOWN_BAD_SESSIONS = {("subj01", 11)}

VAE_MODEL_ID = "mindeye/sd_image_var_autoenc"
VAE_CKPT_URL = "https://huggingface.co/datasets/pscotti/mindeyev2/resolve/main/sd_image_var_autoenc.pth"
VAE_CKPT_PATH = f"{INPUT_DIR}/sd_image_var_autoenc.pth"
VAE_IMAGE_SIZE = 512
VAE_ENCODE_BATCH = 12
VAE_DECODE_BATCH = 32
VAE_CACHE_SHARD_SIZE = 128
COPY_MINDEYE_SD_CACHE_TO_LOCAL = False
MINDEYE_SD_CACHE_DIR = f"{INPUT_DIR}/subject01_mindeye_sd_image_var_autoenc_features_{VAE_IMAGE_SIZE}"
MINDEYE_SD_LOCAL_CACHE_DIR = f"{BASE}/subject01_mindeye_sd_image_var_autoenc_features_{VAE_IMAGE_SIZE}"
MINDEYE_SD_CACHE_METADATA = f"{MINDEYE_SD_CACHE_DIR}/metadata.json"
MINDEYE_SD_LOCAL_CACHE_METADATA = f"{MINDEYE_SD_LOCAL_CACHE_DIR}/metadata.json"
RUN_MINDEYE_SD_CACHE_BUILDER = not (os.path.exists(MINDEYE_SD_CACHE_METADATA) or os.path.exists(MINDEYE_SD_LOCAL_CACHE_METADATA))

D_MODEL = 768
N_LAYERS = 4
N_HEADS = 8
FFN_DIM = 3072
DROPOUT = 0.10
BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-4
GRAD_CLIP_NORM = 1.0
LOWLEVEL_SIDE = 16
LOWLEVEL_CHANNELS = 64

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"device={DEVICE} batch_size={BATCH_SIZE} amp={USE_AMP} output={OUTPUT_DIR}")
print(f"MindEye SD Drive cache metadata: {MINDEYE_SD_CACHE_METADATA} exists={os.path.exists(MINDEYE_SD_CACHE_METADATA)}")
print(f"MindEye SD local cache metadata: {MINDEYE_SD_LOCAL_CACHE_METADATA} exists={os.path.exists(MINDEYE_SD_LOCAL_CACHE_METADATA)}")


## NSD Design + Flatmap Sessions


In [ ]:
import urllib.request
from scipy.io import loadmat

EXP = f"{INPUT_DIR}/nsd_expdesign.mat"
if not os.path.exists(EXP):
    urllib.request.urlretrieve("https://natural-scenes-dataset.s3.amazonaws.com/nsddata/experiments/nsd/nsd_expdesign.mat", EXP)
mat = loadmat(EXP)

masterordering = mat["masterordering"].reshape(-1).astype(np.int64) - 1
subjectim = mat["subjectim"].astype(np.int64) - 1
subject_idx = int(SUBJECT[-2:]) - 1
imgbrick_ids_all = subjectim[subject_idx, masterordering]
shared_ids = set(mat["sharedix"].reshape(-1).astype(np.int64) - 1)

session_files = sorted(glob.glob(f"{FLATMAP_TOKENS_DIR}/{SUBJECT}_flatmap_tokens_session*.pt"))
session_nums = [int(re.search(r"session(\d+)", f).group(1)) for f in session_files]
keep = [(f, s) for f, s in zip(session_files, session_nums) if (SUBJECT, s) not in KNOWN_BAD_SESSIONS]
session_files = [f for f, _ in keep]
session_nums = [s for _, s in keep]
assert len(session_files) >= 2, f"need at least 2 flatmap sessions, found {len(session_files)}"

needed_ids = set()
for sess in session_nums:
    start = (sess - 1) * TRIALS_PER_SESSION
    needed_ids.update(imgbrick_ids_all[start:start + TRIALS_PER_SESSION].astype(int).tolist())
unique_subject_img_ids = np.array(sorted(needed_ids), dtype=np.int64)

val_session_nums = session_nums[-1:]
train_session_nums = session_nums[:-1]
print(f"{SUBJECT}: {len(session_nums)} flatmap sessions available: {session_nums}")
print("train sessions:", train_session_nums)
print("val session:", val_session_nums)
print(f"unique images referenced by flatmap sessions: {len(unique_subject_img_ids)}")


## Load MindEye Frozen SD Autoencoder


In [ ]:
def download_with_progress(url, dst, desc="download"):
    import urllib.request
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = f"{dst}.part"
    if os.path.exists(tmp):
        os.remove(tmp)
    with urllib.request.urlopen(url) as response, open(tmp, "wb") as f:
        total = int(response.headers.get("Content-Length", 0))
        with tqdm(total=total, unit="B", unit_scale=True, desc=desc) as pbar:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                f.write(chunk)
                pbar.update(len(chunk))
    os.replace(tmp, dst)

if not os.path.exists(VAE_CKPT_PATH):
    download_with_progress(VAE_CKPT_URL, VAE_CKPT_PATH, desc="download sd_image_var_autoenc.pth")

vae = AutoencoderKL(
    down_block_types=["DownEncoderBlock2D", "DownEncoderBlock2D", "DownEncoderBlock2D", "DownEncoderBlock2D"],
    up_block_types=["UpDecoderBlock2D", "UpDecoderBlock2D", "UpDecoderBlock2D", "UpDecoderBlock2D"],
    block_out_channels=[128, 256, 512, 512],
    layers_per_block=2,
    sample_size=256,
)
vae.load_state_dict(torch.load(VAE_CKPT_PATH, map_location="cpu"))
vae = vae.to(DEVICE).eval()
vae.requires_grad_(False)
VAE_SCALING_FACTOR = 0.18215
print(f"loaded {VAE_MODEL_ID} from {VAE_CKPT_PATH} scaling_factor={VAE_SCALING_FACTOR}")


## Temporary Cache Builder
Run once. It opens `nsd_stimuli.hdf5`, encodes all subject01 images into MindEye SD autoencoder latents, and saves resized ground-truth images for later eval.


In [ ]:
def copy_with_progress(src, dst, desc, chunk_mb=128):
    total = os.path.getsize(src)
    tmp = f"{dst}.part"
    if os.path.exists(tmp):
        os.remove(tmp)
    with open(src, "rb") as fsrc, open(tmp, "wb") as fdst, tqdm(total=total, unit="B", unit_scale=True, desc=desc) as pbar:
        while True:
            chunk = fsrc.read(chunk_mb * 1024 * 1024)
            if not chunk:
                break
            fdst.write(chunk)
            pbar.update(len(chunk))
    if os.path.getsize(tmp) != total:
        raise IOError(f"Incomplete copy: {tmp}")
    shutil.copystat(src, tmp)
    os.replace(tmp, dst)

def find_stimulus_dataset(h5):
    for key in ["imgBrick", "images", "stimuli"]:
        if key in h5:
            return h5[key]
    keys = list(h5.keys())
    raise KeyError(f"Could not find NSD image dataset. HDF5 keys={keys}")

def preprocess_nsd_uint8(images_np):
    img = torch.from_numpy(images_np).permute(0, 3, 1, 2).float().div(255.0)
    img = F.interpolate(img, size=(VAE_IMAGE_SIZE, VAE_IMAGE_SIZE), mode="bicubic", align_corners=False).clamp(0, 1)
    return img

@torch.no_grad()
def encode_mindeye_sd_images(img01):
    img_m11 = img01.to(DEVICE, non_blocking=True).mul(2).sub(1)
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        posterior = vae.encode(img_m11).latent_dist
        latents = posterior.mode() * VAE_SCALING_FACTOR
    return latents.float().cpu()

def build_mindeye_sd_cache():
    os.makedirs(MINDEYE_SD_CACHE_DIR, exist_ok=True)
    stim_path = f"{BASE}/nsd_stimuli.hdf5"
    src_stim_path = f"{INPUT_DIR}/nsd_stimuli.hdf5"
    if not os.path.exists(stim_path):
        if not os.path.exists(src_stim_path):
            raise FileNotFoundError(f"Missing NSD stimuli HDF5: {src_stim_path}")
        copy_with_progress(src_stim_path, stim_path, "copy nsd_stimuli.hdf5")

    shards = []
    with h5py.File(stim_path, "r") as h5:
        img_ds = find_stimulus_dataset(h5)
        for shard_i, start in enumerate(tqdm(range(0, len(unique_subject_img_ids), VAE_CACHE_SHARD_SIZE), desc="MindEye SD target shards")):
            ids = unique_subject_img_ids[start:start + VAE_CACHE_SHARD_SIZE]
            latents, images_u8 = [], []
            for b0 in range(0, len(ids), VAE_ENCODE_BATCH):
                batch_ids = ids[b0:b0 + VAE_ENCODE_BATCH]
                images_np = img_ds[batch_ids]
                img01 = preprocess_nsd_uint8(images_np)
                latents.append(encode_mindeye_sd_images(img01).half())
                images_u8.append(img01.mul(255).round().byte())
            shard = {
                "img_ids": ids.astype(int).tolist(),
                "latents": torch.cat(latents, dim=0),
                "images_u8": torch.cat(images_u8, dim=0),
            }
            shard_file = f"mindeye_sd_image_var_autoenc_features_{shard_i:05d}.pt"
            torch.save(shard, f"{MINDEYE_SD_CACHE_DIR}/{shard_file}")
            shards.append({"file": shard_file, "img_ids": shard["img_ids"]})
            del shard, latents, images_u8
            gc.collect()

    sample = torch.load(f"{MINDEYE_SD_CACHE_DIR}/{shards[0]['file']}", map_location="cpu")
    metadata = {
        "model_id": VAE_MODEL_ID,
        "image_size": VAE_IMAGE_SIZE,
        "scaling_factor": VAE_SCALING_FACTOR,
        "latent_shape": list(sample["latents"].shape[1:]),
        "num_images": int(len(unique_subject_img_ids)),
        "dtype": "float16",
        "shard_size": VAE_CACHE_SHARD_SIZE,
        "shards": shards,
    }
    with open(f"{MINDEYE_SD_CACHE_DIR}/metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"saved metadata={MINDEYE_SD_CACHE_DIR}/metadata.json")

if RUN_MINDEYE_SD_CACHE_BUILDER:
    print("MindEye SD cache metadata not found in Drive or local runtime; building from nsd_stimuli.hdf5")
    build_mindeye_sd_cache()
elif os.path.exists(MINDEYE_SD_LOCAL_CACHE_METADATA):
    print(f"local MindEye SD cache exists, skipping builder: {MINDEYE_SD_LOCAL_CACHE_DIR}")
else:
    print(f"Drive MindEye SD cache exists, skipping builder: {MINDEYE_SD_CACHE_DIR}")

if COPY_MINDEYE_SD_CACHE_TO_LOCAL and os.path.exists(MINDEYE_SD_CACHE_METADATA) and not os.path.exists(MINDEYE_SD_LOCAL_CACHE_METADATA):
    print(f"copying MindEye SD cache to local disk: {MINDEYE_SD_LOCAL_CACHE_DIR}")
    shutil.copytree(MINDEYE_SD_CACHE_DIR, MINDEYE_SD_LOCAL_CACHE_DIR)
    print(f"copied MindEye SD cache to local disk: {MINDEYE_SD_LOCAL_CACHE_DIR}")
elif os.path.exists(MINDEYE_SD_CACHE_METADATA) and not os.path.exists(MINDEYE_SD_LOCAL_CACHE_METADATA):
    print("using Drive MindEye SD cache directly; set COPY_MINDEYE_SD_CACHE_TO_LOCAL=True to mirror it to /content")


## Load Cached MindEye SD Latents + Flatmap Tokens


In [ ]:
MINDEYE_SD_READ_CACHE_DIR = MINDEYE_SD_LOCAL_CACHE_DIR if os.path.exists(MINDEYE_SD_LOCAL_CACHE_METADATA) else MINDEYE_SD_CACHE_DIR
metadata_path = MINDEYE_SD_LOCAL_CACHE_METADATA if os.path.exists(MINDEYE_SD_LOCAL_CACHE_METADATA) else MINDEYE_SD_CACHE_METADATA
if not os.path.exists(metadata_path):
    raise FileNotFoundError(f"Missing MindEye SD cache metadata: {metadata_path}")

with open(metadata_path) as f:
    mindeye_sd_manifest = json.load(f)

LATENT_SHAPE = tuple(mindeye_sd_manifest["latent_shape"])
LATENT_DIM = int(np.prod(LATENT_SHAPE))
if tuple(LATENT_SHAPE) != (4, 64, 64):
    raise ValueError(f"Expected MindEye SD latents (4,64,64), got {LATENT_SHAPE}.")

NUM_QUERIES = LOWLEVEL_SIDE * LOWLEVEL_SIDE
TARGET_DIM = LOWLEVEL_CHANNELS

print(f"reading MindEye SD targets from {MINDEYE_SD_READ_CACHE_DIR}")
print(f"latent_shape={LATENT_SHAPE} bridge_queries={NUM_QUERIES} bridge_target_dim={TARGET_DIM}")

_mindeye_sd_lookup = {}
for shard in mindeye_sd_manifest["shards"]:
    shard_path = f"{MINDEYE_SD_READ_CACHE_DIR}/{shard['file']}"
    if not os.path.exists(shard_path):
        raise FileNotFoundError(f"Missing MindEye SD shard: {shard_path}")
    for row, img_id in enumerate(shard["img_ids"]):
        _mindeye_sd_lookup[int(img_id)] = (shard_path, row)

_mindeye_sd_lru = OrderedDict()
MINDEYE_SD_CACHE_MAX_SHARDS_IN_RAM = 32

def _load_mindeye_sd_shard(path):
    if path in _mindeye_sd_lru:
        _mindeye_sd_lru.move_to_end(path)
        return _mindeye_sd_lru[path]
    try:
        shard = torch.load(path, map_location="cpu", mmap=True)
    except TypeError:
        shard = torch.load(path, map_location="cpu")
    _mindeye_sd_lru[path] = shard
    while len(_mindeye_sd_lru) > MINDEYE_SD_CACHE_MAX_SHARDS_IN_RAM:
        _mindeye_sd_lru.popitem(last=False)
    return shard

def mindeye_sd_targets_for_ids(img_ids, include_images=False, device=DEVICE):
    ids = [int(i) for i in img_ids.detach().cpu().tolist()]
    missing = [i for i in ids if i not in _mindeye_sd_lookup]
    if missing:
        raise KeyError(f"Missing cached MindEye SD targets for image ids: {missing[:10]}")

    latents = torch.empty((len(ids), *LATENT_SHAPE), dtype=torch.float16)
    images = torch.empty((len(ids), 3, VAE_IMAGE_SIZE, VAE_IMAGE_SIZE), dtype=torch.uint8) if include_images else None

    by_shard = defaultdict(list)
    for out_row, img_id in enumerate(ids):
        shard_path, shard_row = _mindeye_sd_lookup[img_id]
        by_shard[shard_path].append((out_row, shard_row))

    for shard_path, pairs in by_shard.items():
        shard = _load_mindeye_sd_shard(shard_path)
        out_rows = [p[0] for p in pairs]
        shard_rows = torch.as_tensor([p[1] for p in pairs], dtype=torch.long)
        latents[out_rows] = shard["latents"][shard_rows]
        if include_images:
            images[out_rows] = shard["images_u8"][shard_rows]

    out = {"latents": latents.to(device, non_blocking=True).float() if device is not None else latents.float()}
    if include_images:
        out["images"] = images.float().div(255.0)
    return out

LOCAL_FLATMAP_CACHE_DIR = "/content/flatmap_tokens_fp16_cache"
os.makedirs(LOCAL_FLATMAP_CACHE_DIR, exist_ok=True)

# Colab /content is temporary. Leave this False for resume/decode without re-copying.
COPY_FLATMAP_TOKENS_TO_LOCAL = False


def local_cache_path(sess):
    return f"{LOCAL_FLATMAP_CACHE_DIR}/{SUBJECT}_session{sess:02d}_fp16.pt"


def drive_session_path(sess):
    return f"{FLATMAP_TOKENS_DIR}/{SUBJECT}_flatmap_tokens_session{sess:02d}.pt"


def flatmap_read_path(sess):
    local = local_cache_path(sess)
    if os.path.exists(local):
        return local
    src = drive_session_path(sess)
    if os.path.exists(src):
        return src
    raise FileNotFoundError(
        f"Missing flatmap session {sess:02d}. Checked:\n"
        f"  local: {local}\n"
        f"  drive: {src}"
    )


def prepare_local_flatmap_session(sess):
    dest = local_cache_path(sess)
    if os.path.exists(dest):
        return dest
    src = drive_session_path(sess)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing flatmap session tokens: {src}")
    raw_tmp = f"{dest}.raw"
    fp16_tmp = f"{dest}.tmp"
    for tmp_path in [raw_tmp, fp16_tmp]:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
    copy_with_progress(src, raw_tmp, desc=f"copy flatmap session {sess:02d}", chunk_mb=256)
    brain_tokens = torch.load(raw_tmp, map_location="cpu")
    if brain_tokens.dtype != torch.float16:
        brain_tokens = brain_tokens.half()
    torch.save(brain_tokens.contiguous(), fp16_tmp)
    os.replace(fp16_tmp, dest)
    os.remove(raw_tmp)
    del brain_tokens
    gc.collect()
    return dest


def load_flatmap_tokens(sess):
    path = prepare_local_flatmap_session(sess) if COPY_FLATMAP_TOKENS_TO_LOCAL else flatmap_read_path(sess)
    brain_tokens = torch.load(path, map_location="cpu")
    if brain_tokens.dtype != torch.float16:
        brain_tokens = brain_tokens.half()
    return brain_tokens.contiguous()


def ids_for_session(sess, n_rows):
    start = (sess - 1) * TRIALS_PER_SESSION
    return imgbrick_ids_all[start:start + n_rows].astype(np.int64)


def load_session(sess):
    """-> brain_tokens [trials, brain_tokens, D], target_latents [trials, 4, 64, 64]."""
    brain_tokens = load_flatmap_tokens(sess)
    ids = torch.from_numpy(ids_for_session(sess, brain_tokens.shape[0])).long()
    targets = mindeye_sd_targets_for_ids(ids, device=None)["latents"]

    if not torch.isfinite(brain_tokens).all():
        bad = (~torch.isfinite(brain_tokens)).sum().item()
        raise ValueError(f"session {sess}: {bad} non-finite values in brain_tokens")
    if not torch.isfinite(targets).all():
        bad = (~torch.isfinite(targets)).sum().item()
        raise ValueError(f"session {sess}: {bad} non-finite values in MindEye SD targets")

    return brain_tokens, targets

print(
    "flatmap tokens: preparing local fp16 cache on demand"
    if COPY_FLATMAP_TOKENS_TO_LOCAL
    else "flatmap tokens: reading directly from Drive unless local cache already exists"
)


## Flatmap Cross-Attention + MindEye Latent Upsampler


In [ ]:
class CrossAttentionBridge(nn.Module):
    def __init__(self, brain_dim, target_dim, num_queries, d_model, n_layers, n_heads, ffn_dim, dropout):
        super().__init__()
        self.query_embed = nn.Parameter(torch.randn(num_queries, d_model) * 0.02)
        self.brain_proj = nn.Linear(brain_dim, d_model) if brain_dim != d_model else nn.Identity()
        self.brain_norm = nn.LayerNorm(d_model)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, target_dim)

    def forward(self, brain_tokens):
        """brain_tokens: [B, N_brain, brain_dim] -> [B, num_queries, target_dim]."""
        b = brain_tokens.shape[0]
        memory = self.brain_norm(self.brain_proj(brain_tokens))
        queries = self.query_embed.unsqueeze(0).repeat(b, 1, 1)
        out = self.decoder(tgt=queries, memory=memory)
        return self.output_proj(out)

class FlatmapMindEyeSDBridge(nn.Module):
    def __init__(self, brain_dim):
        super().__init__()
        self.bridge = CrossAttentionBridge(
            brain_dim=brain_dim,
            target_dim=LOWLEVEL_CHANNELS,
            num_queries=LOWLEVEL_SIDE * LOWLEVEL_SIDE,
            d_model=D_MODEL,
            n_layers=N_LAYERS,
            n_heads=N_HEADS,
            ffn_dim=FFN_DIM,
            dropout=DROPOUT,
        )
        self.norm = nn.GroupNorm(1, LOWLEVEL_CHANNELS)
        self.upsampler = Decoder(
            in_channels=LOWLEVEL_CHANNELS,
            out_channels=LATENT_SHAPE[0],
            up_block_types=["UpDecoderBlock2D", "UpDecoderBlock2D", "UpDecoderBlock2D"],
            block_out_channels=[64, 128, 256],
            layers_per_block=1,
        )

    def forward(self, brain_tokens, return_seed=False):
        seed_tokens = self.bridge(brain_tokens)
        seed = seed_tokens.transpose(1, 2).reshape(seed_tokens.shape[0], LOWLEVEL_CHANNELS, LOWLEVEL_SIDE, LOWLEVEL_SIDE).contiguous()
        seed = self.norm(seed)
        pred_latents = self.upsampler(seed)
        if return_seed:
            return pred_latents, seed
        return pred_latents

def infer_brain_dim():
    for ckpt_path in [
        f"{OUTPUT_DIR}/mindeye_sd_flatmap_bridge.pt",
        f"{OUTPUT_DIR}/best_mindeye_sd_flatmap_bridge.pt",
    ]:
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location="cpu")
            if isinstance(ckpt, dict) and "brain_dim" in ckpt:
                print(f"brain token dim from checkpoint: {ckpt_path}")
                return int(ckpt["brain_dim"])
    print("brain token dim from first flatmap session")
    return int(load_flatmap_tokens(train_session_nums[0]).shape[-1])

brain_dim = infer_brain_dim()
print("brain token dim:", brain_dim)

model = FlatmapMindEyeSDBridge(brain_dim=brain_dim).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
num_params = sum(p.numel() for p in model.parameters())
print(f"flatmap mindeye-sd bridge params: {num_params / 1e6:.1f}M")


## Train Flatmap-To-MindEye-SD Bridge


In [ ]:
def run_epoch(session_nums_, epoch, train: bool):
    model.train(train)
    totals = defaultdict(float)
    n_batches = 0
    order = list(session_nums_)
    if train:
        random.shuffle(order)

    desc = f"[mindeye-sd bridge] epoch {epoch}/{EPOCHS} ({'train' if train else 'val'})"
    for sess in tqdm(order, desc=desc, leave=False):
        brain_tokens, target_latents = load_session(sess)
        perm = torch.randperm(brain_tokens.shape[0]) if train else torch.arange(brain_tokens.shape[0])

        for b in range(0, len(perm), BATCH_SIZE):
            idx = perm[b:b + BATCH_SIZE]
            brain_batch = brain_tokens[idx].to(DEVICE, non_blocking=True)
            target_batch = target_latents[idx].to(DEVICE, non_blocking=True)

            with torch.set_grad_enabled(train):
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                    pred = model(brain_batch)
                    latent_l1 = F.l1_loss(pred, target_batch)
                    latent_mse = F.mse_loss(pred, target_batch)
                    loss = latent_l1 / VAE_SCALING_FACTOR

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()

            with torch.no_grad():
                latent_cos = F.cosine_similarity(pred.flatten(1), target_batch.flatten(1), dim=1).mean()
            totals["loss"] += float(loss.detach().cpu())
            totals["latent_l1"] += float(latent_l1.detach().cpu())
            totals["latent_mse"] += float(latent_mse.detach().cpu())
            totals["latent_cos"] += float(latent_cos.detach().cpu())
            n_batches += 1

        del brain_tokens, target_latents
        gc.collect()

    return {k: v / max(n_batches, 1) for k, v in totals.items()}

LAST_CKPT_PATH = f"{OUTPUT_DIR}/mindeye_sd_flatmap_bridge.pt"
BEST_CKPT_PATH = f"{OUTPUT_DIR}/best_mindeye_sd_flatmap_bridge.pt"
HISTORY_PATH = f"{OUTPUT_DIR}/loss_history.json"


def checkpoint_payload(epoch):
    return {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "epoch": epoch,
        "best_val": best_val,
        "brain_dim": brain_dim,
        "target_dim": TARGET_DIM,
        "num_queries": NUM_QUERIES,
        "lowlevel_side": LOWLEVEL_SIDE,
        "lowlevel_channels": LOWLEVEL_CHANNELS,
        "latent_shape": LATENT_SHAPE,
        "vae_model_id": VAE_MODEL_ID,
        "vae_ckpt_path": VAE_CKPT_PATH,
        "vae_scaling_factor": VAE_SCALING_FACTOR,
        "vae_image_size": VAE_IMAGE_SIZE,
        "d_model": D_MODEL,
        "n_layers": N_LAYERS,
        "n_heads": N_HEADS,
        "ffn_dim": FFN_DIM,
        "dropout": DROPOUT,
        "train_sessions": train_session_nums,
        "val_sessions": val_session_nums,
        "loss": "MindEye-style F.l1_loss(pred_latents, target_latents) / 0.18215",
    }


def save_progress(epoch, is_best=False):
    payload = checkpoint_payload(epoch)
    torch.save(payload, LAST_CKPT_PATH)
    if is_best:
        torch.save(payload, BEST_CKPT_PATH)
    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=2)

history = []
best_val = float("inf")
start_epoch = 1
if os.path.exists(HISTORY_PATH):
    with open(HISTORY_PATH) as f:
        history = json.load(f)
    if history:
        best_val = min(h["val_loss"] for h in history)
        start_epoch = max(h["epoch"] for h in history) + 1

if os.path.exists(LAST_CKPT_PATH):
    ckpt = torch.load(LAST_CKPT_PATH, map_location=DEVICE)
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    model.load_state_dict(state)
    if isinstance(ckpt, dict) and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    if isinstance(ckpt, dict) and "epoch" in ckpt:
        start_epoch = max(start_epoch, int(ckpt["epoch"]) + 1)
    if isinstance(ckpt, dict) and "best_val" in ckpt:
        best_val = min(best_val, float(ckpt["best_val"]))
    print(f"resuming from {LAST_CKPT_PATH}; next epoch={start_epoch}")

if start_epoch > EPOCHS:
    print(f"training already complete for EPOCHS={EPOCHS}; loaded {len(history)} history rows")
else:
    for epoch in range(start_epoch, EPOCHS + 1):
        train_scores = run_epoch(train_session_nums, epoch, train=True)
        val_scores = run_epoch(val_session_nums, epoch, train=False)
        row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_scores.items()}, **{f"val_{k}": v for k, v in val_scores.items()}}
        history.append(row)
        print(
            f"epoch {epoch:02d} train loss={row['train_loss']:.5f} l1={row['train_latent_l1']:.5f} cos={row['train_latent_cos']:.4f} "
            f"val loss={row['val_loss']:.5f} l1={row['val_latent_l1']:.5f} cos={row['val_latent_cos']:.4f}"
        )
        is_best = row["val_loss"] < best_val
        if is_best:
            best_val = row["val_loss"]
        save_progress(epoch, is_best=is_best)

print(f"saved: {LAST_CKPT_PATH}, {BEST_CKPT_PATH}, {HISTORY_PATH}")

if history:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=180)
    axes[0].plot([h["train_loss"] for h in history], label="train")
    axes[0].plot([h["val_loss"] for h in history], label="val")
    axes[0].set_title("MindEye SD latent L1 / scale")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    axes[1].plot([h["train_latent_cos"] for h in history], label="train")
    axes[1].plot([h["val_latent_cos"] for h in history], label="val")
    axes[1].set_title("MindEye SD latent cosine")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/mindeye_sd_flatmap_bridge_loss.png", dpi=180, bbox_inches="tight")
    plt.show()


## Evaluate Flatmap-To-MindEye-SD Bridge


In [ ]:
BRIDGE_EVAL_PATH = f"{OUTPUT_DIR}/mindeye_sd_flatmap_bridge_eval.json"
BEST_CKPT_PATH = f"{OUTPUT_DIR}/best_mindeye_sd_flatmap_bridge.pt"
LAST_CKPT_PATH = f"{OUTPUT_DIR}/mindeye_sd_flatmap_bridge.pt"

if os.path.exists(BRIDGE_EVAL_PATH):
    with open(BRIDGE_EVAL_PATH) as f:
        summary = json.load(f)
    print(f"bridge eval already exists, skipping: {BRIDGE_EVAL_PATH}")
    print(json.dumps(summary, indent=2))
else:
    ckpt_path = BEST_CKPT_PATH if os.path.exists(BEST_CKPT_PATH) else LAST_CKPT_PATH
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Missing bridge checkpoint: {BEST_CKPT_PATH} or {LAST_CKPT_PATH}")
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt)
    model.eval()
    train_eval_scores = run_epoch(train_session_nums[:3], epoch=0, train=False)
    val_scores = run_epoch(val_session_nums, epoch=0, train=False)
    summary = {"checkpoint": ckpt_path, "train_first3_sessions": train_eval_scores, "val_sessions": val_scores}
    with open(BRIDGE_EVAL_PATH, "w") as f:
        json.dump(summary, f, indent=2)
    print(json.dumps(summary, indent=2))


# 2. Decode Predicted MindEye SD Latents


In [ ]:
@torch.no_grad()
def decode_mindeye_sd_latents(latents):
    latents = latents.to(DEVICE, non_blocking=True) / VAE_SCALING_FACTOR
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        imgs = vae.decode(latents).sample
    return imgs.float().add(1).div(2).clamp(0, 1).cpu()


@torch.no_grad()
def predict_mindeye_sd_latents(brain_tokens):
    model.eval()
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        pred_latents = model(brain_tokens.to(DEVICE, non_blocking=True)).float()
    return pred_latents


SHARED_TOKENS_PATH = f"{OUTPUT_DIR}/mindeye_sd_flatmap_bridge_shared1000_brain_tokens.pt"
IDS_PATH = f"{OUTPUT_DIR}/mindeye_sd_flatmap_bridge_shared1000_img_ids.json"
RECONS_PATH = f"{OUTPUT_DIR}/mindeye_sd_flatmap_bridge_shared1000_recons.pt"
RECON_SHARD_DIR = f"{OUTPUT_DIR}/mindeye_sd_flatmap_bridge_shared1000_recon_shards"
SHARED_SESSION_PART_DIR = f"{OUTPUT_DIR}/mindeye_sd_flatmap_bridge_shared1000_session_parts"
os.makedirs(RECON_SHARD_DIR, exist_ok=True)
os.makedirs(SHARED_SESSION_PART_DIR, exist_ok=True)


def shared_session_part_path(sess):
    return f"{SHARED_SESSION_PART_DIR}/session{sess:02d}.pt"


def collect_shared1000_flatmap_tokens():
    if os.path.exists(SHARED_TOKENS_PATH) and os.path.exists(IDS_PATH):
        print(f"loading cached shared-1000 flatmap tokens: {SHARED_TOKENS_PATH}")
        with open(IDS_PATH) as f:
            ids = json.load(f)
        return torch.load(SHARED_TOKENS_PATH, map_location="cpu"), ids

    shared_id_set = set(int(i) for i in shared_ids)
    for sess in tqdm(session_nums, desc="collect shared-1000 session parts"):
        part_path = shared_session_part_path(sess)
        if os.path.exists(part_path):
            continue
        brain_tokens = load_flatmap_tokens(sess)
        sess_ids = ids_for_session(sess, brain_tokens.shape[0])
        rows = [row for row, img_id in enumerate(sess_ids) if int(img_id) in shared_id_set]
        part = {
            "session": int(sess),
            "img_ids": [int(sess_ids[row]) for row in rows],
            "brain_tokens": brain_tokens[rows].cpu() if rows else torch.empty((0, *brain_tokens.shape[1:]), dtype=brain_tokens.dtype),
        }
        tmp_path = f"{part_path}.tmp"
        torch.save(part, tmp_path)
        os.replace(tmp_path, part_path)
        del brain_tokens, part
        gc.collect()

    groups = defaultdict(list)
    for sess in session_nums:
        part_path = shared_session_part_path(sess)
        if not os.path.exists(part_path):
            raise FileNotFoundError(f"Missing shared session part: {part_path}")
        part = torch.load(part_path, map_location="cpu")
        for img_id, token in zip(part["img_ids"], part["brain_tokens"]):
            groups[int(img_id)].append(token)

    ids = sorted(groups)
    avg_tokens = torch.stack([torch.stack(groups[img_id]).mean(0) for img_id in ids]).half().contiguous()
    tmp_tokens = f"{SHARED_TOKENS_PATH}.tmp"
    torch.save(avg_tokens, tmp_tokens)
    os.replace(tmp_tokens, SHARED_TOKENS_PATH)
    with open(f"{IDS_PATH}.tmp", "w") as f:
        json.dump([int(i) for i in ids], f)
    os.replace(f"{IDS_PATH}.tmp", IDS_PATH)
    print(f"saved shared-1000 flatmap tokens: {SHARED_TOKENS_PATH} shape={tuple(avg_tokens.shape)}")
    return avg_tokens, ids


def recon_shard_path(start, end):
    return f"{RECON_SHARD_DIR}/recons_{start:04d}_{end:04d}.pt"


def decode_shared1000():
    if os.path.exists(RECONS_PATH) and os.path.exists(IDS_PATH):
        print(f"loading cached reconstructions: {RECONS_PATH}")
        with open(IDS_PATH) as f:
            ids = json.load(f)
        return torch.load(RECONS_PATH, map_location="cpu"), ids

    ckpt_path = f"{OUTPUT_DIR}/best_mindeye_sd_flatmap_bridge.pt"
    if not os.path.exists(ckpt_path):
        ckpt_path = f"{OUTPUT_DIR}/mindeye_sd_flatmap_bridge.pt"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Missing model checkpoint in OUTPUT_DIR: {OUTPUT_DIR}")
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt)
    model.eval()

    shared_brain_tokens, ids = collect_shared1000_flatmap_tokens()

    for start in tqdm(range(0, len(shared_brain_tokens), BATCH_SIZE), desc="decode shared-1000"):
        end = min(start + BATCH_SIZE, len(shared_brain_tokens))
        shard_path = recon_shard_path(start, end)
        if os.path.exists(shard_path):
            continue
        pred_latents = predict_mindeye_sd_latents(shared_brain_tokens[start:end])
        batch_recons = []
        for d0 in range(0, len(pred_latents), VAE_DECODE_BATCH):
            batch_recons.append(decode_mindeye_sd_latents(pred_latents[d0:d0 + VAE_DECODE_BATCH]))
        shard = torch.cat(batch_recons, dim=0).clamp(0, 1)
        tmp_path = f"{shard_path}.tmp"
        torch.save(shard, tmp_path)
        os.replace(tmp_path, shard_path)
        del pred_latents, batch_recons, shard
        gc.collect()

    shards = []
    for start in range(0, len(shared_brain_tokens), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(shared_brain_tokens))
        shard_path = recon_shard_path(start, end)
        if not os.path.exists(shard_path):
            raise FileNotFoundError(f"Missing reconstruction shard: {shard_path}")
        shards.append(torch.load(shard_path, map_location="cpu"))
    recons = torch.cat(shards, dim=0).clamp(0, 1)
    tmp_recons = f"{RECONS_PATH}.tmp"
    torch.save(recons, tmp_recons)
    os.replace(tmp_recons, RECONS_PATH)
    print(f"saved={RECONS_PATH} shape={tuple(recons.shape)}")
    return recons, ids


all_recons, shared_img_ids = decode_shared1000()
print("all_recons", tuple(all_recons.shape), "shared_img_ids", len(shared_img_ids))


# 3. Shared-1000 Reconstruction Eval


In [ ]:
from skimage.color import rgb2gray
from skimage.metrics import structural_similarity as ssim_fn

RECON_EVAL_PATH = f"{OUTPUT_DIR}/mindeye_sd_flatmap_bridge_shared1000_eval.csv"

if os.path.exists(RECON_EVAL_PATH):
    metrics_df = pd.read_csv(RECON_EVAL_PATH, index_col=0)
    print(f"reconstruction eval already exists, skipping: {RECON_EVAL_PATH}")
    display(metrics_df)
else:
    all_recons = torch.load(RECONS_PATH, map_location="cpu").float().clamp(0, 1)
    with open(IDS_PATH) as f:
        shared_img_ids = json.load(f)
    EVAL_N = min(1000, len(all_recons), len(shared_img_ids))
    recons_eval = all_recons[:EVAL_N]
    ids_eval = torch.tensor(shared_img_ids[:EVAL_N], dtype=torch.long)

    @torch.no_grad()
    def eval_reconstruction_metrics(recons, img_ids, batch_size=64):
        pixel_mse_total = 0.0
        pixcorr_scores = []
        ssim_scores = []
        latent_mse_total = 0.0
        latent_cos_total = 0.0
        seen = 0
        for start in tqdm(range(0, len(recons), batch_size), desc="eval recons"):
            rec = recons[start:start + batch_size].float()
            batch_ids = img_ids[start:start + batch_size]
            cache = mindeye_sd_targets_for_ids(batch_ids, include_images=True)
            true_img = cache["images"].float()
            n = len(rec)
            pixel_mse_total += n * float(F.mse_loss(rec, true_img).cpu())
            r = rec.flatten(1).numpy()
            t = true_img.flatten(1).numpy()
            pixcorr_scores.extend(float(np.corrcoef(t[i], r[i])[0, 1]) for i in range(n))
            rec_gray = rgb2gray(rec.permute(0, 2, 3, 1).numpy())
            true_gray = rgb2gray(true_img.permute(0, 2, 3, 1).numpy())
            ssim_scores.extend(ssim_fn(rec_gray[i], true_gray[i], data_range=1.0, gaussian_weights=True, sigma=1.5, use_sample_covariance=False) for i in range(n))

            rec_latents = encode_mindeye_sd_images(rec)
            true_latents = cache["latents"].detach().cpu()
            latent_mse_total += n * float(F.mse_loss(rec_latents, true_latents).cpu())
            latent_cos_total += n * float(F.cosine_similarity(rec_latents.flatten(1), true_latents.flatten(1), dim=1).mean().cpu())
            seen += n
        return {
            "N": seen,
            "Pixel_MSE": pixel_mse_total / seen,
            "PixCorr": float(np.mean(pixcorr_scores)),
            "SSIM": float(np.mean(ssim_scores)),
            "MindEyeSD_Latent_MSE": latent_mse_total / seen,
            "MindEyeSD_Latent_Cosine": latent_cos_total / seen,
        }

    metrics = eval_reconstruction_metrics(recons_eval, ids_eval)
    metrics_df = pd.DataFrame([metrics]).set_index("N")
    display(metrics_df)
    metrics_df.to_csv(RECON_EVAL_PATH)
    print(f"saved={RECON_EVAL_PATH}")


## Preview Grid


In [ ]:
PREVIEW_GRID_N = 48
PREVIEW_GRID_COLS = 6
PREVIEW_GRID_SEED = 42
PREVIEW_GRID_PATH = f"{OUTPUT_DIR}/mindeye_sd_flatmap_bridge_preview_grid.png"

if os.path.exists(PREVIEW_GRID_PATH):
    print(f"preview grid already exists, skipping: {PREVIEW_GRID_PATH}")
else:
    all_recons = torch.load(RECONS_PATH, map_location="cpu").float().clamp(0, 1)
    with open(IDS_PATH) as f:
        shared_img_ids = json.load(f)
    n_show = min(PREVIEW_GRID_N, len(all_recons))
    rng = np.random.RandomState(PREVIEW_GRID_SEED)
    show_idx = rng.choice(len(all_recons), size=n_show, replace=False)
    cols = min(PREVIEW_GRID_COLS, n_show)
    rows = int(np.ceil(n_show / cols))

    true_images = mindeye_sd_targets_for_ids(torch.tensor([shared_img_ids[int(i)] for i in show_idx]), include_images=True)["images"].float()
    pred_images = all_recons[show_idx].float()
    combined = torch.cat([true_images, pred_images], dim=3).clamp(0, 1)

    fig, axes = plt.subplots(rows, cols, figsize=(4.8 * cols, 3.0 * rows), constrained_layout=True)
    axes = np.atleast_1d(axes).reshape(rows, cols)
    for ax in axes.ravel():
        ax.axis("off")
    for k, idx in enumerate(show_idx):
        ax = axes[k // cols, k % cols]
        ax.imshow(combined[k].permute(1, 2, 0))
        ax.set_title(f"shared id {shared_img_ids[int(idx)]}\nactual | predicted", fontsize=8)
        ax.axis("off")
    plt.savefig(PREVIEW_GRID_PATH, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"saved={PREVIEW_GRID_PATH}")
